# Build Your First Voice Agent using Exotel

This cookbook shows how to build a **real-time voice agent that answers phone calls**, using **[Exotel](https://exotel.com)** for telephony, **[Pipecat](https://docs.pipecat.ai)** to orchestrate the real-time audio pipeline, and **[Sarvam AI](https://docs.sarvam.ai)** for speech-to-text, the LLM, and text-to-speech.

It's a great starting point for IVR replacements, customer support lines, and conversational phone agents in Indian languages.

By the end of this notebook you will have an agent that can:

- Answer inbound phone calls on an Exotel number
- Listen to callers speaking in multiple Indian languages
- Understand and process what they say
- Respond back in a natural-sounding voice, over the phone

> **Note on running this notebook:** A phone voice agent is a long-running WebSocket server that Exotel streams call audio to — it isn't something you can "run to completion" in a notebook cell. This notebook builds the agent file for you (`agent.py`) using `%%writefile`, and walks through the Exotel console configuration (App Bazaar flow, ngrok tunnel) you need alongside it.

## Overview

| | |
|---|---|
| **Telephony** | [Exotel](https://my.exotel.com) (Voice Streaming over WebSocket) |
| **Pipeline orchestration** | [Pipecat](https://docs.pipecat.ai) |
| **Speech-to-text** | Sarvam **Saarika/Saaras** (`SarvamSTTService`) |
| **LLM** | Sarvam chat models (`SarvamLLMService`, default `sarvam-105b`) |
| **Text-to-speech** | Sarvam **Bulbul v3** (`SarvamTTSService`) |
| **Languages** | 10 Indian languages + English, with auto-detect |
| **Lines of code** | ~80 |

### Quick overview of the steps

1. Get a Sarvam API key. Exotel needs no API credentials for this flow.
2. Install `pipecat-ai` with the `websocket` and `sarvam` extras.
3. Create a `.env` file with your API key.
4. Write the agent (`agent.py`).
5. Point an Exotel Voicebot Applet at your agent's WebSocket URL.
6. Call your Exotel number.

## Prerequisites

- Python 3.9 or higher.
- [ngrok](https://ngrok.com/download), to expose your local server during development.
- An [Exotel](https://my.exotel.com) account with voice streaming enabled and a provisioned ExoPhone.
- A [Sarvam AI](https://dashboard.sarvam.ai) API key.

> Unlike Twilio, Exotel's dial-in flow doesn't require an account SID or auth token in your bot code. Pipecat's `ExotelFrameSerializer` only needs the stream and call identifiers Exotel sends over the WebSocket itself.

## 1. Install dependencies

The `websocket` extra adds Pipecat's WebSocket/FastAPI transport (used to accept Exotel's Voice Streaming connection); the `sarvam` extra adds the Sarvam STT/LLM/TTS services.

In [ ]:
%pip install -Uqq "pipecat-ai[websocket,sarvam]" python-dotenv loguru

## 2. Configure your Sarvam API key

Get your key from the [Sarvam dashboard](https://dashboard.sarvam.ai). Never commit real API keys to version control — the cell below writes a `.env.example` template; copy it to `.env` and fill in your real key locally:

```bash
cp .env.example .env
```

In [ ]:
%%writefile .env.example
# Copy this file to `.env` and fill in your real credentials.
# Do not commit `.env` to version control.

SARVAM_API_KEY=your_sarvam_api_key

## 3. Write the voice agent

Same Pipecat pipeline shape as the Twilio integration (transport in → STT → LLM → TTS → transport out) but with Exotel's transport, which needs no account credentials — just the stream/call identifiers Exotel sends when it connects.

`%%writefile` saves this cell's contents to `agent.py` in the current directory.

In [ ]:
%%writefile agent.py
import os

from dotenv import load_dotenv
from loguru import logger
from pipecat.frames.frames import LLMRunFrame
from pipecat.pipeline.pipeline import Pipeline
from pipecat.pipeline.runner import PipelineRunner
from pipecat.pipeline.task import PipelineParams, PipelineTask
from pipecat.processors.aggregators.llm_context import LLMContext
from pipecat.processors.aggregators.llm_response_universal import (
    LLMContextAggregatorPair,
)
from pipecat.runner.types import RunnerArguments
from pipecat.runner.utils import create_transport
from pipecat.services.sarvam.llm import SarvamLLMService
from pipecat.services.sarvam.stt import SarvamSTTService
from pipecat.services.sarvam.tts import SarvamTTSService
from pipecat.transports.websocket.fastapi import FastAPIWebsocketParams

load_dotenv(override=True)


async def bot(runner_args: RunnerArguments) -> None:
    """Main bot entry point."""

    # create_transport auto-detects Exotel's WebSocket handshake and builds the
    # matching ExotelFrameSerializer using the stream and call identifiers Exotel
    # sends when it connects. No Exotel credentials are needed here.
    transport = await create_transport(
        runner_args,
        {
            "exotel": lambda: FastAPIWebsocketParams(
                audio_in_enabled=True, audio_out_enabled=True
            ),
        },
    )

    # Initialize the Sarvam AI services.
    stt = SarvamSTTService(api_key=os.getenv("SARVAM_API_KEY"))
    tts = SarvamTTSService(api_key=os.getenv("SARVAM_API_KEY"))
    llm = SarvamLLMService(
        api_key=os.getenv("SARVAM_API_KEY"),
        settings=SarvamLLMService.Settings(model="sarvam-105b"),
    )

    # Set up conversation context.
    messages = [
        {
            "role": "system",
            "content": "You are a friendly AI phone assistant. Keep your responses brief and conversational.",
        },
    ]
    context = LLMContext(messages)
    context_aggregator = LLMContextAggregatorPair(context)

    # Build the pipeline: audio in -> STT -> LLM -> TTS -> audio out.
    pipeline = Pipeline(
        [
            transport.input(),
            stt,
            context_aggregator.user(),
            llm,
            tts,
            transport.output(),
            context_aggregator.assistant(),
        ]
    )

    # Exotel Voice Streaming sends and receives 8kHz mono audio by default.
    task = PipelineTask(
        pipeline,
        params=PipelineParams(
            audio_in_sample_rate=8000,
            audio_out_sample_rate=8000,
        ),
    )

    @transport.event_handler("on_client_connected")
    async def on_client_connected(transport, client) -> None:
        logger.info("Caller connected")
        messages.append(
            {"role": "system", "content": "Greet the caller and briefly introduce yourself."}
        )
        await task.queue_frames([LLMRunFrame()])

    @transport.event_handler("on_client_disconnected")
    async def on_client_disconnected(transport, client) -> None:
        logger.info("Caller disconnected")
        await task.cancel()

    runner = PipelineRunner(handle_sigint=runner_args.handle_sigint)
    await runner.run(task)


if __name__ == "__main__":
    from pipecat.runner.run import main

    main()

## 4. Configure Exotel to reach your agent

Exotel is different from Twilio here: it doesn't fetch an XML webhook at call time. Instead, you paste your WebSocket URL directly into a **Voicebot Applet** inside a call flow, and Exotel opens that connection itself.

**Start ngrok** (in a separate terminal):

```bash
ngrok http 7860
```

ngrok prints a forwarding URL such as `https://xxxx.ngrok-free.app`. Turn it into a `wss://` URL for the next step.

> The domain suffix ngrok assigns varies per account (`ngrok-free.app`, `ngrok-free.dev`, or the older `ngrok.io`) — always use whatever URL ngrok actually prints.
>
> Free ngrok URLs change every time you restart ngrok. If your agent stops receiving calls, check whether the URL in your Voicebot Applet still matches the one ngrok is currently printing.

**Build the App Bazaar flow:**

1. In the [Exotel dashboard](https://my.exotel.com), go to **App Bazaar**.
2. Create a new custom app (or edit an existing one).
3. Add a **Voicebot** applet (not **Stream** or **Passthru**), and set its URL field to `wss://xxxx.ngrok-free.app/ws`.
4. Add a **Hangup** applet right after it, so the flow is: **Call Start → Voicebot Applet → Hangup Applet**.
5. Save the flow.

For production use, Exotel recommends adding a **Passthru** applet between Voicebot and Hangup to capture session metadata (call SID, duration, recording URL) and detect disconnects. It's optional for getting a call working.

**Assign the flow to your number:**

1. Go to **ExoPhones**, and select your number.
2. Under **Installed App**, choose the flow you just built.
3. Confirm the change when prompted.

## 5. Run and test your agent

```bash
python agent.py --transport exotel
```

No `--proxy` flag is needed here since Exotel doesn't fetch a webhook — it connects straight to `/ws`.

Call your Exotel number from any phone — your voice agent will pick up and start the conversation.

## Customization examples

Swap these blocks into the service-initialization section of `agent.py` and re-run.

> `language`, `voice`, and `target_language_code` are `Settings` fields, not constructor keyword arguments. Passing them directly to `SarvamSTTService(...)` or `SarvamTTSService(...)` raises a `TypeError` on current versions of `pipecat-ai`. Only `mode` (STT) and `api_key` stay outside `settings`.

### Hindi voice agent

```python
stt = SarvamSTTService(
    api_key=os.getenv("SARVAM_API_KEY"),
    settings=SarvamSTTService.Settings(model="saaras:v3", language="hi-IN"),
    mode="transcribe",
)

tts = SarvamTTSService(
    api_key=os.getenv("SARVAM_API_KEY"),
    settings=SarvamTTSService.Settings(
        model="bulbul:v3",
        voice="simran",  # or: priya, ishita, kavya, aditya, anand, rohan
        target_language_code="hi-IN",
    ),
)

llm = SarvamLLMService(
    api_key=os.getenv("SARVAM_API_KEY"),
    settings=SarvamLLMService.Settings(model="sarvam-105b"),
)
```

### Tamil voice agent

```python
stt = SarvamSTTService(
    api_key=os.getenv("SARVAM_API_KEY"),
    settings=SarvamSTTService.Settings(model="saaras:v3", language="ta-IN"),
    mode="transcribe",
)

tts = SarvamTTSService(
    api_key=os.getenv("SARVAM_API_KEY"),
    settings=SarvamTTSService.Settings(
        model="bulbul:v3", voice="shubh", target_language_code="ta-IN"
    ),
)
```

### Multilingual agent (auto-detect)

Use `language="unknown"` for support lines where callers might speak any of several languages. Sarvam auto-detects the spoken language per utterance, so the same agent can handle a Hindi caller followed by a Tamil caller without any code changes.

```python
stt = SarvamSTTService(
    api_key=os.getenv("SARVAM_API_KEY"),
    settings=SarvamSTTService.Settings(model="saaras:v3", language="unknown"),
    mode="transcribe",
)

tts = SarvamTTSService(
    api_key=os.getenv("SARVAM_API_KEY"),
    settings=SarvamTTSService.Settings(
        model="bulbul:v3", voice="anand", target_language_code="en-IN"
    ),
)
```

### Speech-to-English agent (Saaras)

Saarika transcribes speech to text in the same language. Saaras translates speech directly to English text. Use Saaras when the caller speaks an Indian language but you want to process and respond in English — it auto-detects the source language (Hindi, Tamil, etc.) and translates spoken content directly to English text.

```python
stt = SarvamSTTService(
    api_key=os.getenv("SARVAM_API_KEY"),
    settings=SarvamSTTService.Settings(model="saaras:v3"),
    mode="translate",  # speech-to-English translation
)

tts = SarvamTTSService(
    api_key=os.getenv("SARVAM_API_KEY"),
    settings=SarvamTTSService.Settings(
        model="bulbul:v3", voice="aditya", target_language_code="en-IN"
    ),
)

llm = SarvamLLMService(
    api_key=os.getenv("SARVAM_API_KEY"),
    settings=SarvamLLMService.Settings(model="sarvam-105b"),
)
```

## Available options

### Language codes

| Language | Code |
|---|---|
| English (India) | `en-IN` |
| Hindi | `hi-IN` |
| Bengali | `bn-IN` |
| Tamil | `ta-IN` |
| Telugu | `te-IN` |
| Gujarati | `gu-IN` |
| Kannada | `kn-IN` |
| Malayalam | `ml-IN` |
| Marathi | `mr-IN` |
| Punjabi | `pa-IN` |
| Odia | `od-IN` |
| Auto-detect | `unknown` |

### Speaker voices (Bulbul v3)

- **Male (23):** shubh (default), aditya, rahul, rohan, amit, dev, ratan, varun, manan, sumit, kabir, aayan, ashutosh, advait, anand, tarun, sunny, mani, gokul, vijay, mohit, rehan, soham
- **Female (14):** ritu, priya, neha, pooja, simran, kavya, ishita, shreya, roopa, tanya, shruti, suhani, kavitha, rupali

### TTS additional parameters

```python
tts = SarvamTTSService(
    api_key=os.getenv("SARVAM_API_KEY"),
    settings=SarvamTTSService.Settings(
        model="bulbul:v3",
        voice="shubh",
        target_language_code="en-IN",
        pace=1.0,  # range: 0.5 to 2.0
    ),
)
```

> Exotel Voice Streaming defaults to 8kHz mono audio when the Voicebot Applet's WebSocket URL doesn't specify a rate (this guide's URL doesn't). Exotel also supports 16kHz and 24kHz via a `?sample-rate=` query parameter on that URL — if you use one, update `audio_in_sample_rate`/`audio_out_sample_rate` in `PipelineParams` to match. You don't need to touch `sample_rate` on `SarvamTTSService` either way: Pipecat resamples the TTS output to match `audio_out_sample_rate` automatically.

## Understanding the call flow

```mermaid
flowchart LR
    caller(["Caller"]) <--> exotel["Exotel"]
    exotel <-->|"wss:// audio"| pipeline

    subgraph pipeline["Your Agent Server (Pipecat)"]
        direction LR
        stt["STT"] --> llm["LLM"] --> tts["TTS"]
    end

    pipeline -.->|HTTPS| sarvam["Sarvam AI\nSaarika/Saaras · sarvam-30b/105b · Bulbul"]
```

1. **Caller dials your Exotel number.** The App Bazaar flow reaches the Voicebot Applet, and Exotel opens a WebSocket directly to your agent — there's no webhook fetch in between.
2. **Transport input** — your Pipecat transport receives the caller's audio over the WebSocket and hands it to STT.
3. **STT** — converts audio to text using Sarvam's Saarika or Saaras; the context aggregator adds it to the conversation context.
4. **LLM** — generates a response using Sarvam.
5. **TTS** — converts the response to audio using Sarvam's Bulbul.
6. **Transport output** — streams the audio back over the WebSocket, and Exotel plays it to the caller, while the context aggregator saves the assistant's response to context. The **Hangup Applet** ends the call once your agent closes the WebSocket.

## Pro tips

- Use `language="unknown"` to automatically detect the caller's language — works well for multilingual support lines.
- Sarvam's models understand code-mixing, so your agent can naturally handle Hinglish, Tanglish, and other mixed languages, which is common on real support calls.
- Use `sarvam-30b` for faster responses, or `sarvam-105b` for more complex conversations.
- Log the transcript from `context_aggregator` if you want post-call analytics — writing it to a file or database is cheap and won't slow down the live pipeline.

## Troubleshooting

- **Call connects but there's no audio** — Confirm the Voicebot Applet's URL uses `wss://`, not `ws://` or `https://`, and that it points at your current ngrok URL. Free ngrok URLs change on every restart.
- **Call doesn't reach your agent at all** — Double-check the flow order in App Bazaar: **Call Start → Voicebot Applet → Hangup Applet**. A missing or misordered applet is the most common cause.
- **API key errors** — Make sure `SARVAM_API_KEY` is in your `.env` file and that the file sits in the same directory as `agent.py`.
- **Module not found** — Re-run the install cell for your operating system.
- **Poor transcription** — Try `language="unknown"` for auto-detection, or set the exact language code (`en-IN`, `hi-IN`, etc.) if you already know it.
- **Choppy or robotic audio** — Make sure `audio_in_sample_rate` and `audio_out_sample_rate` in `PipelineParams` match the rate Exotel is actually streaming: `8000` by default, or whatever you set via `?sample-rate=` on the Voicebot Applet's WebSocket URL (Exotel also supports 16000 and 24000). A mismatch causes distorted playback.
- **Call disconnects a few seconds after connecting** — Exotel expects your WebSocket server to accept the connection within about 10 seconds. Make sure ngrok and `agent.py` are both already running before you place the call.
- **Long call cuts off unexpectedly** — Exotel caps a single streaming session at around 60 minutes (varies by plan) — design your agent to end the conversation before that limit for calls that need to run longer.

## Additional resources

- [Sarvam AI Documentation](https://docs.sarvam.ai)
- [Pipecat Documentation](https://docs.pipecat.ai)
- [Pipecat Sarvam LLM Service](https://docs.pipecat.ai/api-reference/server/services/llm/sarvam)
- [Exotel Stream and Voicebot Applet Guide](https://support.exotel.com/support/solutions/articles/3000108630-working-with-the-stream-and-voicebot-applet)
- [Exotel Updated Extension Guide (timeouts, session limits, Passthru)](https://support.exotel.com/support/solutions/articles/3000132302-updated-extension-guide-working-with-the-stream-and-voicebot-applet-beta-)
- [Exotel Dashboard](https://my.exotel.com)

## Need help?

- Sarvam Support: developer@sarvam.ai
- Community: [Join the Discord Community](https://discord.com/invite/5rAsykttcs)